# grads-dict-accumulate-parents — worked example 1: grads dict: first visit vs revisit for parent tensors

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `grads-dict-accumulate-parents`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """Minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries optional `.recipe`,
    `.requires_grad`, and `.grad` (the accumulated gradient at leaves)."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Concept

During backpropagation, the reverse-pass accumulator `grads` is a dictionary keyed by tensor objects (identity, not value). The pattern `grads[parent] = grads.get(parent, 0) + g` handles both cases: if `parent` is not yet in the dict, `.get(parent, 0)` returns 0 (the additive identity), seeding the first contribution; on revisits it adds to the running sum. Using `+` (not `+=`) produces a new tensor, avoiding accidental mutation of any gradient a caller might be holding.

## Worked solution

**Step 1 — create two parent node objects.** We simulate two parent tensors in a compute graph. Each will receive one gradient contribution in this example.

**Step 2 — first contributions.** Call the accumulation function with each parent's first contribution. The dict starts empty, so `.get(parent, 0)` returns 0 and the result is simply the contribution itself.

**Step 3 — revisit one parent.** The first parent receives a second contribution (simulating a shared node). Now `.get` finds the existing entry and adds to it.

**Step 4 — read final values.** After all contributions, `grads[p1]` is the sum of both p1 contributions; `grads[p2]` is just its one contribution.

**Step 5 — verify no mutation.** Save a reference before the second accumulation and confirm the saved tensor was not modified in place.

In [ ]:
import torch as t

t.manual_seed(0)

class Node:
    """Minimal stand-in for a compute graph node."""
    def __init__(self, name):
        self.name = name

def accumulate_into_grads(grads, contributions):
    for parent, g in contributions:
        grads[parent] = grads.get(parent, 0) + g

# Two parent nodes
p1 = Node('w1')
p2 = Node('w2')

grads = {}

# First contributions
g1a = t.tensor([1.0, 2.0])
g2a = t.tensor([3.0, -1.0])
accumulate_into_grads(grads, [(p1, g1a), (p2, g2a)])
print(f"After first round: p1={grads[p1]}, p2={grads[p2]}")

# Save reference before second p1 contribution
ref_p1 = grads[p1]

# Second contribution to p1 (simulating two backward paths through p1)
g1b = t.tensor([0.5, 0.5])
accumulate_into_grads(grads, [(p1, g1b)])
print(f"After second p1 contribution: p1={grads[p1]}, p2={grads[p2]}")

assert t.allclose(grads[p1], g1a + g1b)
assert t.allclose(grads[p2], g2a)
assert t.allclose(ref_p1, g1a), "Old reference must not be mutated (should use +, not +=)"
print("All checks passed.")